# DroneAI runtime probe

Mount Google Drive, create persistent artifact folders, and record the Colab environment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/DroneAI')
for name in ('datasets', 'checkpoints', 'runs'):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)

In [ ]:
import json, os, platform, subprocess, sys
from datetime import datetime, timezone

import tensorflow as tf
import torch

def command_output(command):
    try:
        return subprocess.run(command, check=True, capture_output=True, text=True).stdout.strip()
    except Exception as exc:
        return f'{type(exc).__name__}: {exc}'

report = {
    'captured_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'colab_release_tag': os.environ.get('COLAB_RELEASE_TAG'),
    'nvidia_smi': command_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader']),
    'torch': {
        'version': torch.__version__,
        'cuda_runtime': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(),
        'device_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    },
    'tensorflow': {
        'version': tf.__version__,
        'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
    },
}
output = DRIVE_ROOT / 'runs' / 'runtime-probe' / 'environment.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
print(f'Saved: {output}')